# 05A — RSNA Knee Dense-Slice MIL — Submit

Minimal offline inference notebook. It requires only the competition input and the attached `05A_dense_mil_final.pt`. It never trains and creates exactly `/kaggle/working/submission.csv`.


In [ ]:
# 1. Imports, constants, and hard runtime guard
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from collections import Counter
from time import monotonic
import json, math, os, warnings

import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torchvision.models import resnet18

warnings.filterwarnings("ignore")
START_TIME = monotonic()
HARD_DEADLINE_SECONDS = 8 * 3600
SAFE_TARGET_SECONDS = 7 * 3600
SMOKE_TEST = False
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SLOTS = ["SAG_FLUID", "SAG_NONFLUID", "COR_FLUID", "COR_NONFLUID", "AX_FLUID"]
ROUTING = {"SAG_FLUID": ("Sagittal", 1), "SAG_NONFLUID": ("Sagittal", 0),
           "COR_FLUID": ("Coronal", 1), "COR_NONFLUID": ("Coronal", 0), "AX_FLUID": ("Axial", 1)}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
STATS = Counter(); DECODE_TRANSFER_SYNTAX_FAILURES = Counter()
print("device:", DEVICE, "GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])


## 2. Shallow offline path resolution and checkpoint validation


In [ ]:
def shallow_files(name, max_depth=5):
    root = Path("/kaggle/input"); ans = []
    if not root.exists(): return ans
    for current, dirs, files in os.walk(root):
        current_path = Path(current); depth = len(current_path.relative_to(root).parts)
        dirs[:] = [d for d in dirs if d not in {"train_series", "test_series"}]
        if depth >= max_depth: dirs[:] = []
        if name in files: ans.append(current_path / name)
    print(f"Candidates for {name}:", [str(p) for p in ans])
    return sorted(set(ans), key=lambda p: (len(p.parts), str(p)))

def resolve_paths():
    candidates = [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                  Path("/kaggle/input/rsna-knee-abnormality-detection")]
    candidates += [p.parent for p in shallow_files("test.csv")]
    comp = next((p for p in candidates if all((p / x).exists() for x in ["test.csv", "test_series.csv", "test_series"])), None)
    if comp is None: raise FileNotFoundError("Competition test input not found at expected shallow paths")
    checkpoints = shallow_files("05A_dense_mil_final.pt")
    if not checkpoints: raise FileNotFoundError("Attach training output containing 05A_dense_mil_final.pt")
    return comp, checkpoints[0]

COMP_ROOT, CHECKPOINT_PATH = resolve_paths()
try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
except TypeError:  # compatibility with older Kaggle PyTorch images
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")
CFG = checkpoint["config"]
assert checkpoint["labels"] == LABELS and checkpoint["slots"] == SLOTS
assert not bool(CFG.get("smoke_test", False)), "Refusing to submit a SMOKE_TEST checkpoint"
assert CFG["backbone"] == "resnet18" and CFG["backbone_weights"] is None
assert int(CFG["image_size"]) == 224 and float(CFG["physical_crop_mm"]) == 140.0
print("competition:", COMP_ROOT); print("checkpoint:", CHECKPOINT_PATH); print(json.dumps(CFG, indent=2))


## 3. Reproduce notebook 03's deterministic test-series selection


In [ ]:
TEST_DICOM_ROOT = COMP_ROOT / "test_series"
test = pd.read_csv(COMP_ROOT / "test.csv", dtype={"StudyInstanceUID": str})
series = pd.read_csv(COMP_ROOT / "test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})

def dcm_files(path):
    try: return [Path(e.path) for e in os.scandir(path) if e.is_file() and e.name.lower().endswith(".dcm")]
    except Exception: return []

def sf(x):
    try: return float(x)
    except Exception: return np.nan

def one_series_metadata(row):
    files = dcm_files(TEST_DICOM_ROOT / str(row.StudyInstanceUID) / str(row.SeriesInstanceUID))
    out = {"n_slices": len(files), "PixelSpacingMean": np.nan, "FOVRow_mm": np.nan, "FOVCol_mm": np.nan}
    if not files: return out
    try:
        ds = pydicom.dcmread(str(files[0]), stop_before_pixels=True, force=True)
        ps = getattr(ds, "PixelSpacing", None); rs = sf(ps[0]) if ps is not None and len(ps) >= 2 else np.nan
        cs = sf(ps[1]) if ps is not None and len(ps) >= 2 else np.nan
        rows, cols = sf(getattr(ds, "Rows", np.nan)), sf(getattr(ds, "Columns", np.nan))
        out = {"n_slices": len(files), "PixelSpacingMean": np.nanmean([rs, cs]),
               "FOVRow_mm": rows * rs, "FOVCol_mm": cols * cs}
    except Exception: STATS["selection_header_failures"] += 1
    return out

with ThreadPoolExecutor(max_workers=8) as ex:
    meta = list(ex.map(one_series_metadata, series.itertuples(index=False)))
series = pd.concat([series.reset_index(drop=True), pd.DataFrame(meta)], axis=1)

def route(row):
    try: key = (str(row.Anatomical_Plane), int(row.Fluid_Sensitive))
    except Exception: return None
    for name, value in ROUTING.items():
        if key == value: return name
    return None

series["slot"] = series.apply(route, axis=1)
series["usable_slice_count"] = series.n_slices.between(12, 100).astype(int)
series["fov_ok"] = (series[["FOVRow_mm", "FOVCol_mm"]].min(axis=1) >= 140.0).astype(int)
series["spacing_rank_value"] = series.PixelSpacingMean.fillna(999.0)
series["slice_count_distance"] = (series.n_slices - 32).abs()
ranked = series.sort_values(["StudyInstanceUID", "slot", "usable_slice_count", "fov_ok", "spacing_rank_value",
                             "slice_count_distance", "SeriesInstanceUID"],
                            ascending=[True, True, False, False, True, True, True])
selected = ranked.dropna(subset=["slot"]).groupby(["StudyInstanceUID", "slot"], as_index=False).first()
test_manifest = test[["StudyInstanceUID"]].copy()
for slot in SLOTS:
    part = selected[selected.slot.eq(slot)][["StudyInstanceUID", "SeriesInstanceUID"]].rename(
        columns={"SeriesInstanceUID": f"{slot}__SeriesInstanceUID"})
    test_manifest = test_manifest.merge(part, on="StudyInstanceUID", how="left", validate="one_to_one")
assert list(test_manifest.StudyInstanceUID) == list(test.StudyInstanceUID)
print("hidden test studies:", len(test_manifest))


## 4. Geometry-safe DICOM decoding and variable-budget preprocessing


In [ ]:
def sort_dicom_series(series_dir):
    records = []
    for path in dcm_files(series_dir):
        try:
            ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
            iop, ipp = getattr(ds, "ImageOrientationPatient", None), getattr(ds, "ImagePositionPatient", None)
            position = np.nan
            if iop is not None and ipp is not None and len(iop) >= 6 and len(ipp) >= 3:
                a, b = np.asarray(iop[:6], float), np.asarray(ipp[:3], float); normal = np.cross(a[:3], a[3:])
                if np.isfinite(a).all() and np.isfinite(b).all() and np.linalg.norm(normal) > 1e-8:
                    position = float(np.dot(b, normal))
            records.append((path, position, sf(getattr(ds, "InstanceNumber", np.nan))))
        except Exception: STATS["failed_headers"] += 1
    if not records: return []
    if all(np.isfinite(x[1]) for x in records): records.sort(key=lambda x: x[1]); STATS["geometry_sorted_series"] += 1
    elif all(np.isfinite(x[2]) for x in records): records.sort(key=lambda x: x[2]); STATS["instance_fallback_series"] += 1
    else: STATS["failed_series_order"] += 1; return []
    return [x[0] for x in records]

def sample_slice_indices(n, budget):
    if n <= budget: return np.arange(n, dtype=np.int32)
    return np.unique(np.rint(np.linspace(0, n - 1, budget)).astype(np.int32))

def read_dicom_slice(path):
    transfer = "unknown"
    try:
        ds = pydicom.dcmread(str(path), force=True)
        transfer = str(getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", "unknown"))
        arr = np.asarray(ds.pixel_array, np.float32).squeeze()
        if arr.ndim != 2: raise ValueError("not 2D")
        arr = arr * sf(getattr(ds, "RescaleSlope", 1.0)) + sf(getattr(ds, "RescaleIntercept", 0.0))
        finite = np.isfinite(arr)
        if not finite.any(): raise ValueError("non-finite")
        arr[~finite] = np.median(arr[finite])
        if str(getattr(ds, "PhotometricInterpretation", "")).upper() == "MONOCHROME1": arr = arr.max() + arr.min() - arr
        lo, hi = np.percentile(arr, [1, 99]); arr = np.zeros_like(arr) if hi <= lo + 1e-6 else np.clip((np.clip(arr, lo, hi) - lo) / (hi - lo), 0, 1)
        ps = getattr(ds, "PixelSpacing", None)
        if ps is None or len(ps) < 2: raise ValueError("PixelSpacing")
        rs, cs = float(ps[0]), float(ps[1])
        if not (np.isfinite(rs) and np.isfinite(cs) and rs > 0 and cs > 0): raise ValueError("invalid PixelSpacing")
        h, w = arr.shape
        ch, cw = min(h, max(1, round(140.0 / rs))), min(w, max(1, round(140.0 / cs)))
        y0, x0 = max(0, (h - ch) // 2), max(0, (w - cw) // 2); arr = arr[y0:y0+ch, x0:x0+cw]
        t = torch.from_numpy(arr)[None, None].float()
        return torch.nn.functional.interpolate(t, size=(224, 224), mode="bilinear", align_corners=False)[0, 0].clamp(0, 1)
    except Exception:
        STATS["failed_pixel_decodes"] += 1; DECODE_TRANSFER_SYNTAX_FAILURES[transfer] += 1; return None

def load_study(row, budget):
    images, slots, positions = [], [], []
    uid = str(row["StudyInstanceUID"])
    for slot_id, slot in enumerate(SLOTS):
        series_uid = row.get(f"{slot}__SeriesInstanceUID", np.nan)
        if pd.isna(series_uid): continue
        ordered = sort_dicom_series(TEST_DICOM_ROOT / uid / str(series_uid))
        for idx in sample_slice_indices(len(ordered), budget):
            im = read_dicom_slice(ordered[int(idx)])
            if im is None: continue
            images.append(im); slots.append(slot_id); positions.append(float(idx / max(1, len(ordered) - 1)))
    return {"uid": uid, "images": images, "slots": slots, "positions": positions}

def collate_studies(items):
    b, t = len(items), max(1, max(len(x["images"]) for x in items))
    images = torch.zeros((b, t, 3, 224, 224)); slots = torch.zeros((b, t), dtype=torch.long)
    positions = torch.zeros((b, t)); mask = torch.zeros((b, t), dtype=torch.bool)
    for i, x in enumerate(items):
        n = len(x["images"])
        if n:
            gray = torch.stack(x["images"]); images[i, :n] = gray[:, None].expand(-1, 3, -1, -1)
            slots[i, :n] = torch.tensor(x["slots"]); positions[i, :n] = torch.tensor(x["positions"]); mask[i, :n] = True
    return [x["uid"] for x in items], images, slots, positions, mask


## 5. Checkpoint-compatible DenseSliceMIL model


In [ ]:
class DenseSliceMIL(nn.Module):
    def __init__(self):
        super().__init__(); backbone = resnet18(weights=None); backbone.fc = nn.Identity(); self.encoder = backbone
        d = int(CFG["token_dim"])
        self.slice_projection = nn.Sequential(nn.Linear(512, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(0.1))
        self.slot_embedding = nn.Embedding(len(SLOTS), d)
        self.position_embedding = nn.Sequential(nn.Linear(1, d), nn.GELU(), nn.Linear(d, d))
        self.queries = nn.Parameter(torch.randn(len(LABELS), d) / math.sqrt(d))
        self.attention = nn.MultiheadAttention(d, int(CFG["attention_heads"]), batch_first=True, dropout=0.1)
        self.classifiers = nn.ModuleList([nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 1)) for _ in LABELS])
    def forward(self, images, slots, positions, mask):
        b, t = mask.shape; d = int(CFG["token_dim"]); tokens = images.new_zeros((b, t, d))
        if mask.any():
            z = self.slice_projection(self.encoder(images[mask]))
            tokens[mask] = z + self.slot_embedding(slots[mask]) + self.position_embedding(positions[mask].unsqueeze(-1))
        pooled = images.new_zeros((b, len(LABELS), d)); nonempty = mask.any(1)
        if nonempty.any():
            q = self.queries[None].expand(int(nonempty.sum()), -1, -1)
            po = self.attention(q, tokens[nonempty], tokens[nonempty], key_padding_mask=~mask[nonempty], need_weights=False)[0]
            pooled[nonempty] = po.to(dtype=pooled.dtype)
        return torch.cat([head(pooled[:, j]).reshape(b, 1) for j, head in enumerate(self.classifiers)], 1)

model = DenseSliceMIL().to(DEVICE); model.load_state_dict(checkpoint["state_dict"], strict=True); model.eval()
if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
print("checkpoint loaded; final training epochs:", checkpoint.get("final_epochs"))


## 6. Runtime benchmark and adaptive 16→12→8 slice budget


In [ ]:
@torch.inference_mode()
def infer_indices(indices, budget, progress=False):
    outputs, uids = [], []; batch_size = 2
    for start in range(0, len(indices), batch_size):
        batch_idx = indices[start:start + batch_size]
        items = [load_study(test_manifest.iloc[i], budget) for i in batch_idx]
        buids, images, slots, positions, mask = collate_studies(items)
        images, slots, positions, mask = [x.to(DEVICE, non_blocking=True) for x in (images, slots, positions, mask)]
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP): logits = model(images, slots, positions, mask)
        outputs.append(torch.sigmoid(logits).cpu().numpy()); uids.extend(buids)
        if progress and (start + len(batch_idx)) % 20 == 0:
            elapsed = monotonic() - START_TIME; done = start + len(batch_idx)
            rate = elapsed / max(1, done); remaining = rate * (len(indices) - done)
            print(f"processed {done}/{len(indices)} elapsed={elapsed/60:.1f}m sec/study={rate:.2f} estimated_remaining={remaining/3600:.2f}h")
    return uids, np.concatenate(outputs)

benchmark_n = min(20 if not SMOKE_TEST else min(2, len(test_manifest)), len(test_manifest))
benchmark_indices = list(range(benchmark_n)); selected_budget = None; benchmark_predictions = None
for budget in [16, 12, 8]:
    t0 = monotonic(); buids, bpred = infer_indices(benchmark_indices, budget); seconds = monotonic() - t0
    sec_per_study = seconds / max(1, benchmark_n)
    projected = (monotonic() - START_TIME) + sec_per_study * len(test_manifest) + 30 * 60
    print(f"budget={budget} benchmark_sec/study={sec_per_study:.3f} projected_total_hours={projected/3600:.3f}")
    if projected <= SAFE_TARGET_SECONDS or budget == 8:
        selected_budget, benchmark_predictions = budget, bpred; break
print("hidden test studies:", len(test_manifest), "selected slice budget:", selected_budget,
      "benchmark sec/study:", sec_per_study, "projected runtime hours:", projected / 3600)


## 7. Full inference with deadline-aware degradation for remaining studies


In [ ]:
predictions = np.empty((len(test_manifest), len(LABELS)), np.float32)
predictions[:benchmark_n] = benchmark_predictions
current_budget = selected_budget; processed = benchmark_n; batch_size = 2
while processed < len(test_manifest):
    elapsed = monotonic() - START_TIME
    observed_rate = elapsed / max(1, processed)
    projected_finish = elapsed + observed_rate * (len(test_manifest) - processed)
    if projected_finish > HARD_DEADLINE_SECONDS - 10 * 60 and current_budget > 8:
        current_budget = 12 if current_budget == 16 else 8
        print("Deadline guard reduced remaining-study slice budget to", current_budget)
    idx = list(range(processed, min(len(test_manifest), processed + batch_size)))
    _, pp = infer_indices(idx, current_budget)
    predictions[processed:processed + len(idx)] = pp; processed += len(idx)
    if processed % 20 == 0 or processed == len(test_manifest):
        elapsed = monotonic() - START_TIME; rate = elapsed / processed
        print(f"processed {processed}/{len(test_manifest)} elapsed={elapsed/60:.1f}m sec/study={rate:.2f} "
              f"estimated_remaining={rate*(len(test_manifest)-processed)/3600:.2f}h budget={current_budget}")


## 8. Exact submission schema and integrity assertions


In [ ]:
submission = pd.DataFrame(predictions, columns=LABELS)
submission.insert(0, "StudyInstanceUID", test.StudyInstanceUID.astype(str).to_numpy())
assert len(submission) == len(test)
assert list(submission.StudyInstanceUID) == list(test.StudyInstanceUID.astype(str))
assert list(submission.columns) == ["StudyInstanceUID"] + LABELS and submission.shape[1] == 13
assert submission.StudyInstanceUID.is_unique and submission.StudyInstanceUID.notna().all()
assert np.isfinite(predictions).all() and predictions.min() >= 0 and predictions.max() <= 1
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("shape:", submission.shape); display(submission.head())
print("probability min/max:", float(predictions.min()), float(predictions.max()))
print("NaN count:", int(submission[LABELS].isna().sum().sum()))
print("DICOM stats:", dict(STATS)); print("decode failure transfer syntaxes:", dict(DECODE_TRANSFER_SYNTAX_FAILURES))
print("total runtime hours:", (monotonic() - START_TIME) / 3600)
